# Feature Engineering on Primary Land Use Tax Lot Output (PLUTO) and High Volume For-Hire Vehicle (HVFHV) Demand Datasets:

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import geopandas as gpd
from shapely import wkt
import pandas as pd 
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_pluto+demand")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/23 21:53:31 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.13.225.117 instead (on interface en0)
24/08/23 21:53:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/23 21:53:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Files:

In [3]:
base_dir = "../data"

PLUTO dataset:

In [4]:
pluto_df_path = base_dir + '/developed/merged_data/pluto_df.csv'
pluto_df = pd.read_csv(pluto_df_path)
pluto_df.head()

,building_class,geometry,location_id,service_zone,zone,borough
0,F,POINT (-74.0750604 40.6288711),221,Boro Zone,Stapleton,Staten Island
1,B,POINT (-74.0746779 40.6154356),221,Boro Zone,Stapleton,Staten Island
2,B,POINT (-74.074487 40.6155098),221,Boro Zone,Stapleton,Staten Island
3,B,POINT (-74.0745662 40.6154741),221,Boro Zone,Stapleton,Staten Island
4,B,POINT (-74.0750307 40.6152734),221,Boro Zone,Stapleton,Staten Island


Zone dataset:

In [5]:
zone_gdf_path = base_dir + '/developed/merged_data/zone_gdf.csv'
zone_gdf = pd.read_csv(zone_gdf_path)
zone_gdf['geometry'] = zone_gdf['geometry'].apply(wkt.loads)
zone_gdf.head()

,location_id,service_zone,Shape_Leng,Shape_Area,zone,borough,geometry
0,1,EWR,0.116357,0.000782,Newark Airport,EWR,POLYGON ((-74.18445299999996 40.69499600000009...
1,2,Boro Zone,0.433470,0.004866,Jamaica Bay,Queens,MULTIPOLYGON (((-73.82337597260663 40.63898704...
2,3,Boro Zone,0.084341,0.000314,Allerton/Pelham Gardens,Bronx,"POLYGON ((-73.84792614099985 40.8713422340001,..."
3,4,Yellow Zone,0.043567,0.000112,Alphabet City,Manhattan,POLYGON ((-73.97177410965318 40.72582128133726...
4,5,Boro Zone,0.092146,0.000498,Arden Heights,Staten Island,POLYGON ((-74.17421738099989 40.56256808600009...


Daily pickup demand:

In [6]:
daily_pickup_demand_sdf_path = base_dir + '/developed/merged_data/daily_pickup_demand'
daily_pickup_demand_sdf = spark.read.parquet(daily_pickup_demand_sdf_path)
daily_pickup_demand_df = daily_pickup_demand_sdf.toPandas()
daily_pickup_demand_df.head()

,pickup_date,PULocationID,daily_demand
0,2023-07-01,2,0.016304
1,2023-07-01,3,6.190217
2,2023-07-01,4,12.043478
3,2023-07-01,5,0.983696
4,2023-07-01,6,1.815217


Daily dropoff demand:

In [7]:
daily_dropoff_demand_sdf_path = base_dir + '/developed/merged_data/daily_dropoff_demand'
daily_dropoff_demand_sdf = spark.read.parquet(daily_dropoff_demand_sdf_path)
daily_dropoff_demand_df = daily_dropoff_demand_sdf.toPandas()
daily_dropoff_demand_df.head()

,pickup_date,DOLocationID,daily_demand
0,2023-07-01,1,25.608696
1,2023-07-01,2,0.021739
2,2023-07-01,3,6.206522
3,2023-07-01,4,9.548913
4,2023-07-01,5,0.918478


# Find Daily Demand by Building Class:

In [23]:
# Merge `pluto_df` and `daily_pickup_demand_df` on `location_id` and `PULocationID`
daily_pickup_demand_by_building_class_df = pd.merge(pluto_df, daily_pickup_demand_df, 
                                             left_on='location_id', 
                                             right_on='PULocationID', 
                                             how='left')

# Group by `building_class`, and calculate the sum of `daily_demand`
daily_pickup_demand_by_building_class_df = daily_pickup_demand_by_building_class_df.groupby(['building_class'])['daily_demand'] \
                                                                     .sum() \
                                                                     .reset_index()

# Sort by daily demand
daily_pickup_demand_by_building_class_df = daily_pickup_demand_by_building_class_df.sort_values(by='daily_demand', ascending=False)\
                                                                     .reset_index(drop=True)

daily_pickup_demand_by_building_class_df.head()

,building_class,daily_demand
0,B,5.171086e+08
1,A,4.298425e+08
2,C,3.926477e+08
3,S,9.472546e+07
4,D,4.750691e+07


In [24]:
daily_pickup_demand_by_building_class_df.head(24)

,building_class,daily_demand
0,B,5.171086e+08
1,A,4.298425e+08
2,C,3.926477e+08
3,S,9.472546e+07
4,D,4.750691e+07
5,K,4.723132e+07
6,V,4.014891e+07
7,R,3.394026e+07
8,G,3.097934e+07
9,O,2.120702e+07


# Find Average Daily Demand by Location ID:

In [8]:
zone_gdf = gpd.GeoDataFrame(zone_gdf, geometry='geometry')
zone_gdf = zone_gdf.drop_duplicates('location_id')

### Pickup:

In [9]:
# Merge `zone_gdf` and `daily_demand_df` on `location_id` and `PULocationID`
daily_demand_by_location_df = pd.merge(zone_gdf, daily_pickup_demand_df, 
                                       left_on='location_id', 
                                       right_on='PULocationID', 
                                       how='left')

# Drop the 'PULocationID' column as it is no longer needed
daily_demand_by_location_df = daily_demand_by_location_df.drop('PULocationID', axis=1)

# Group by `location_id` and calculate the sum of `daily_demand`
daily_demand_by_location_df = daily_demand_by_location_df.groupby(['location_id'])['daily_demand'] \
                                                         .sum() \
                                                         .reset_index()

# Sort the DataFrame by `daily_demand` in descending order
daily_pickup_demand_by_location_df = daily_demand_by_location_df.sort_values(by='daily_demand', ascending=False) \
                                                         .reset_index(drop=True)

daily_pickup_demand_by_location_df.head()

,location_id,daily_demand
0,138,10761.804348
1,132,10112.038043
2,79,7616.304348
3,61,7402.119565
4,230,6427.972826


### Dropoff:

In [27]:
# Merge `zone_gdf` and `daily_dropoff_demand_df` on `location_id` and `DOLocationID`
daily_demand_by_location_df = pd.merge(zone_gdf, daily_dropoff_demand_df, 
                                       left_on='location_id', 
                                       right_on='DOLocationID', 
                                       how='left')

# Drop the 'DOLocationID' column as it is no longer needed
daily_demand_by_location_df = daily_demand_by_location_df.drop('DOLocationID', axis=1)

# Group by `location_id` and calculate the sum of `daily_demand`
daily_demand_by_location_df = daily_demand_by_location_df.groupby(['location_id'])['daily_demand'] \
                                                         .sum() \
                                                         .reset_index()

# Sort the DataFrame by `daily_demand` in descending order
daily_dropoff_demand_by_location_df = daily_demand_by_location_df.sort_values(by='daily_demand', ascending=False) \
                                                         .reset_index(drop=True)

daily_dropoff_demand_by_location_df.head()

,location_id,daily_demand
0,132,15080.815217
1,138,13835.298913
2,61,7743.902174
3,68,6500.304348
4,79,6363.163043


# Save the Merged Dataset:

Daily demand by different building classes:

In [28]:
demand_by_building_class_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_demand_by_building_class_df.csv'
demand_by_building_class_df_path = os.path.join(demand_by_building_class_df_dir, file_name)
daily_pickup_demand_by_building_class_df.to_csv(demand_by_building_class_df_path, index=False)

Daily pickup demand by different location ID:

In [29]:
demand_by_location_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_pickup_demand_by_location.csv'
demand_by_location_df_path = os.path.join(demand_by_location_df_dir, file_name)
daily_pickup_demand_by_location_df.to_csv(demand_by_location_df_path, index=False)

Daily dropoff demand by different location ID:

In [30]:
demand_by_location_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_dropoff_demand_by_location.csv'
demand_by_location_df_path = os.path.join(demand_by_location_df_dir, file_name)
daily_dropoff_demand_by_location_df.to_csv(demand_by_location_df_path, index=False)